# 05 · Label leaks and data traps (checks)

**Goal:** check every property of the data that could reveal the label (same / different entity) *without* reading the names. Anything that does must never become a feature, or the model looks good for the wrong reason.

**How:** for each candidate, how well it predicts the label **on its own**, measured as AUC (0.5 = no better than chance, 1.0 = perfect). For categories, AUC uses each category's share of positives as the score (slightly optimistic, fine for a warning signal).

**Data:** `C.PAIRS_INPUT` (currently the uncleaned pairs). Re-run after the cleaning. Runtime about 4 minutes. Kernel: *Python (ds_sanctions)*.

| # | Candidate | Why it could leak |
| --- | --- | --- |
| 1 | same source list on both sides | merging happens across lists, not within one |
| 2 | both IDs still exist in `targets.simple` | merged records get a new ID, the old ones disappear |
| 3 | form of the ID: `NK-…`, Wikidata `Q…`, or a source ID like `fr-…` | IDs change when records are merged |
| 4 | self-pairs and duplicate pairs | data errors (cleaning) |
| 5 | number of aliases | depends on the source list |
| 6 | pairs per entity, size of the connected component | how often an entity was compared reflects the labelling process |
| 7 | which fields are present on both sides | depends on the source list |
| 8 | name frequency from `targets.simple` | counted after the merge: a merged pair counts once, an unmerged pair twice |

In [ ]:
import sys, re
sys.path.insert(0, "../src")
from collections import Counter
import numpy as np
import pandas as pd
from unidecode import unidecode
from sklearn.metrics import roc_auc_score
from screening import config as C

cols = ["pair_id", "judgement", "left_id", "right_id", "left_schema", "left_datasets", "right_datasets",
        "left_name", "right_name", "left_alias", "right_alias"] + [
        f"{s}_{f}" for s in ("left", "right") for f in ("birthDate", "country", "idNumber", "nationality")]
df = pd.read_parquet(C.PAIRS_INPUT, columns=cols)
df["y"] = (df["judgement"] == "positive").astype(int)
print(f"{C.PAIRS_INPUT.name}: {len(df):,} pairs, {df['y'].mean():.1%} positive")

L = lambda v: [] if v is None else list(v)

def auc(x, y):
    a = roc_auc_score(y, x)
    return round(max(a, 1 - a), 3)

def by_category(x, y):
    t = pd.DataFrame({"category": x, "y": y})
    score = t.groupby("category")["y"].transform("mean")
    table = (t.groupby("category")["y"].agg(pairs="size", share_positive="mean")
               .sort_values("pairs", ascending=False))
    return auc(score, y), table.round(3)

## 1 · Same source list

**What it shows:** share of pairs whose two records share at least one source list (`datasets`), the share of positives in each group, and how often the rule "same list ⇒ different entity" is right.

**How to read it:** a large gap in `share positive` between the two groups, or an AUC far above 0.5, means this property alone reveals the label.

In [ ]:
same = np.array([bool(set(L(a)) & set(L(b))) for a, b in zip(df["left_datasets"], df["right_datasets"])])
rule_acc = ((same & (df["y"] == 0)) | (~same & (df["y"] == 1))).mean()
print(f"same source list: {same.mean():.1%} of pairs; accuracy of 'same list => different': {rule_acc:.1%}; AUC {auc(same.astype(int), df['y'])}")
pd.DataFrame({"pairs": [same.sum(), (~same).sum()],
              "share positive": [df["y"][same].mean(), df["y"][~same].mean()]},
             index=["same source list", "different lists"]).round(3)

## 2 · IDs still present in targets.simple

**What it shows:** share of positives among pairs where both IDs still exist as entities in `targets.simple`, and among the rest.

**How to read it:** when OpenSanctions merges two records, the merged entity gets a new ID. A big gap means "does the ID still exist" reveals whether the pair was merged, i.e. the label.

In [ ]:
tg_ids = set(pd.read_csv(C.DATA_RAW / "targets.simple-20261111.csv", usecols=["id"], dtype=str)["id"])
both = (df["left_id"].isin(tg_ids) & df["right_id"].isin(tg_ids)).to_numpy()
print(f"AUC {auc(both.astype(int), df['y'])}")
pd.DataFrame({"pairs": [both.sum(), (~both).sum()],
              "share positive": [df["y"][both].mean(), df["y"][~both].mean()]},
             index=["both IDs in targets.simple", "at least one missing"]).round(3)

## 3 · Form of the ID

**What it shows:** pairs grouped by the kind of ID on the left and right: `NK` (OpenSanctions entity ID), `Q` (Wikidata ID) or `source-id` (an ID from the original list, e.g. `fr-…`, `us-…`), with the share of positives per combination.

**How to read it:** combinations with a share of positives near 0 or 1 reveal the label from the ID alone.

In [ ]:
def id_kind(s):
    k = s.str.extract(r"^([A-Za-z]+)")[0]
    return k.where(k.isin(["NK", "Q"]), "source-id")
a, table = by_category(id_kind(df["left_id"]) + " / " + id_kind(df["right_id"]), df["y"])
print(f"AUC {a}")
table

## 4 · Self-pairs and duplicate pairs

**What it shows:** pairs with the same ID on both sides, and pairs that occur more than once (same two IDs, in either order), including groups whose copies carry different labels.

**How to read it:** self-pairs are trivially "same entity" and inflate the positives; duplicates count the same judgement several times. Both belong in the cleaning.

In [ ]:
self_pair = df["left_id"] == df["right_id"]
key = pd.Series([tuple(sorted(x)) for x in zip(df["left_id"], df["right_id"])], index=df.index)
dup = key.duplicated(keep=False)
conflicting = (df[dup].groupby(key[dup])["y"].nunique() > 1).sum()
pd.Series({"self-pairs": self_pair.sum(), "share positive among self-pairs": round(df["y"][self_pair].mean(), 3),
           "rows that occur more than once": dup.sum(), "groups of duplicates": key[dup].nunique(),
           "groups with conflicting labels": conflicting}).to_frame("value")

## 5 · Number of aliases

**What it shows:** AUC of the number of aliases (smaller side, larger side, difference) as a predictor of the label.

**How to read it:** clearly above 0.5 means the number of aliases, which mainly depends on the source list, carries label information. Not used as a feature.

In [ ]:
la = df["left_alias"].map(lambda v: len(L(v)))
ra = df["right_alias"].map(lambda v: len(L(v)))
pd.Series({"min of the two": auc(np.minimum(la, ra), df["y"]), "max of the two": auc(np.maximum(la, ra), df["y"]),
           "difference": auc((la - ra).abs(), df["y"])}).to_frame("AUC")

## 6 · Pairs per entity and connected components

**What it shows:** AUC of how many pairs an entity takes part in (larger of the two) and of the size of the connected component the pair belongs to; share of positives inside the giant component vs the rest.

**How to read it:** how often an entity was compared is a property of the labelling process, not of the names. A high AUC means it must not be a feature, and that evaluation should report the giant component separately.

In [ ]:
degree = pd.concat([df["left_id"], df["right_id"]]).value_counts()
deg_max = np.maximum(df["left_id"].map(degree), df["right_id"].map(degree))
parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
for a_, b_ in zip(df["left_id"], df["right_id"]):
    parent[find(a_)] = find(b_)
comp = df["left_id"].map(find)
comp_size = comp.map(comp.value_counts())
giant = comp_size == comp_size.max()
pd.Series({"AUC pairs per entity": auc(deg_max, df["y"]), "AUC component size": auc(comp_size, df["y"]),
           "giant component: share of pairs": round(giant.mean(), 3),
           "giant component: share positive": round(df["y"][giant].mean(), 3),
           "rest: share positive": round(df["y"][~giant].mean(), 3)}).to_frame("value")

## 7 · Which fields are present on both sides

**What it shows:** pairs grouped by which of four fields are present on **both** sides: B = birthDate, C = country, I = idNumber, N = nationality, "-" = missing on at least one side (e.g. `BC--` = birth date and country present). Share of positives per pattern, and AUC overall and for persons / companies.

**How to read it:** part of this is real information (a birth date on both sides allows a strong check), part reflects the source list. An AUC far above 0.5 means record features that encode "missing" as a category can pick up this pattern; compare models with and without the missing categories.

In [ ]:
pattern = pd.Series("", index=df.index)
for f in ["birthDate", "country", "idNumber", "nationality"]:
    present = (df[f"left_{f}"].map(lambda v: len(L(v)) > 0) & df[f"right_{f}"].map(lambda v: len(L(v)) > 0))
    pattern = pattern + np.where(present, f[0].upper(), "-")
a, table = by_category(pattern, df["y"])
person = df["left_schema"] == "Person"
print(f"AUC all {a}; persons {by_category(pattern[person], df['y'][person])[0]}; companies {by_category(pattern[~person], df['y'][~person])[0]}")
table.head(15)

## 8 · Merge effect in the name frequency

**What it shows:** for pairs that share a full name, the number of entities in `targets.simple` that carry the rarest shared name (entities collapsed by name + birth date, as in notebook 03), and the share of positives per count. "0" means the name does not occur in `targets.simple` at all.

**How to read it:** a merged pair (positive) is one entity in `targets.simple`, an unmerged pair (negative) is two, so counts 1 and 2 partly encode the label; "0" (old names of merged records) does too. This is why the feature uses bands with 0, 1 and 2 together. Above that the trend is real: common names are more often different people.

In [ ]:
_NON_ALNUM = re.compile(r"[^a-z0-9]+")
def key_of(name):
    return " ".join(sorted(_NON_ALNUM.sub(" ", unidecode(name or "").lower()).split()))

tt = pd.read_csv(C.DATA_RAW / "targets.simple-20261111.csv", usecols=["id", "schema", "name", "aliases", "birth_date"],
                 dtype=str, keep_default_na=False)
tt = tt[tt["schema"].isin(["Person", "Company", "Organization", "LegalEntity"])]
tt["identity"] = np.where(tt["birth_date"].str.strip() != "", tt["name"].map(key_of) + "|" + tt["birth_date"], "id" + tt["id"])
tt = tt.drop_duplicates(["schema", "identity"])
counts = {"person": Counter(), "org": Counter()}
for sch, name, aliases in zip(tt["schema"], tt["name"], tt["aliases"]):
    g = "person" if sch == "Person" else "org"
    counts[g].update({key_of(v) for v in [name] + (aliases.split(";") if aliases else []) if key_of(v)})

rarest = []
for r in df.itertuples(index=False):
    shared = ({key_of(n) for n in L(r.left_name) + L(r.left_alias)} & {key_of(n) for n in L(r.right_name) + L(r.right_alias)}) - {""}
    g = "person" if r.left_schema == "Person" else "org"
    rarest.append(min(counts[g].get(s, 0) for s in shared) if shared else -1)
t = pd.DataFrame({"n": rarest, "y": df["y"]}).query("n >= 0")
t["entities carrying the name"] = t["n"].clip(upper=6).map(lambda v: "6+" if v == 6 else str(v))
t.groupby("entities carrying the name")["y"].agg(pairs="size", share_positive="mean").round(3)

## Summary (2026-11-16, uncleaned pairs)

| # | Check | Result | Handling |
| --- | --- | --- | --- |
| 1 | same source list | AUC 0.86; 27.7% vs 95.0% positive | never a feature |
| 2 | IDs still in targets.simple | AUC 0.83; 21.1% vs 87.6% positive | never a feature; no lookup of entity data by ID |
| 3 | form of the ID | **AUC 0.92**; source-id/source-id 0.9% positive | never a feature |
| 4 | self-pairs / duplicates | 38,217 self-pairs (all positive); 37,346 duplicate rows in 8,088 groups, 10 with conflicting labels | cleaning |
| 5 | number of aliases | AUC 0.63–0.67 | not a feature |
| 6 | pairs per entity / component size | AUC 0.84 / 0.80; giant component 39% vs 94% positive | not a feature; split by entity; report giant component separately |
| 7 | fields present on both sides | AUC 0.79 (persons 0.80) | record features: compare with and without "missing" categories |
| 8 | name frequency, merge effect | 1 entity 99.1% positive, 2 entities 86.1%, not found 98.8% | bands with 0, 1 and 2 together |

The 92% accuracy for the same-source rule quoted in the project brief is not reproduced here (84.1%).